# Phase 1: Environment Setup

This step installs all required libraries for the project, including:
- SHAP (for explainability)
- imbalanced-learn (for SMOTE balancing)

Run only once. Restart kernel if prompted.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("All libraries loaded!")

## Import Libraries

All essential libraries for:
- Data processing (pandas, numpy)
- Visualization (matplotlib, seaborn)
- Machine Learning (scikit-learn, TensorFlow)
- Explainability (SHAP)

Warnings are suppressed for cleaner output.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("All libraries loaded!")

# Phase 2: Load Dataset

Loads the RTA dataset from the data folder.

Displays:
- Dataset shape
- Column names
- Target variable distribution (Accident Severity)

In [ ]:
df = pd.read_csv('data/RTA Dataset.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nSeverity classes:")
print(df['Accident_severity'].value_counts())
df.head(3)

## Data Cleaning & Encoding

- Selects relevant features for prediction
- Removes missing values
- Converts categorical text data into numerical format using Label Encoding

Result: Clean dataset ready for modeling

In [ ]:
features = [
    'Accident_severity',
    'Day_of_week',
    'Age_band_of_driver',
    'Sex_of_driver',
    'Educational_level',
    'Vehicle_driver_relation',
    'Driving_experience',
    'Type_of_vehicle',
    'Owner_of_vehicle',
    'Service_year_of_vehicle',
    'Area_accident_occured',
    'Lanes_or_Medians',
    'Road_allignment',
    'Types_of_Junction',
    'Road_surface_type',
    'Road_surface_conditions',
    'Light_conditions',
    'Weather_conditions',
    'Type_of_collision',
    'Number_of_vehicles_involved',
    'Number_of_casualties',
    'Cause_of_accident'
]

df = df[features].dropna().reset_index(drop=True)

# Encode all text columns into numbers
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col].astype(str))

print(f"Clean dataset: {df.shape}")
print("\nSeverity distribution after cleaning:")
print(df['Accident_severity'].value_counts())

## Severity Distribution Visualization

Creates a bar chart showing class imbalance:
- Fatal
- Minor Injury
- Serious Injury

This helps justify the need for SMOTE later.

In [ ]:
label_map = {0:'Fatal', 1:'Minor', 2:'Serious'}

plt.figure(figsize=(6, 3.5))
counts = df['Accident_severity'].value_counts().sort_index()
plt.bar([label_map.get(i, i) for i in counts.index],
        counts.values, color=['#e74c3c','#3498db','#f39c12'])
plt.title('Accident Severity Distribution — RTA Dataset')
plt.xlabel('Severity')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('outputs/severity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Phase 3: Data Preparation

Steps:
- Split dataset into features (X) and target (y)
- Normalize features using StandardScaler
- Train-test split (80-20)
- Apply SMOTE to balance class distribution

Result: Balanced dataset for training

In [ ]:
X = df.drop('Accident_severity', axis=1)
y = df['Accident_severity']

# Scale features so the neural network trains properly
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# SMOTE balances the 3 severity classes so model treats all equally
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print(f"Training samples after SMOTE: {X_train_sm.shape[0]}")
print(f"Class counts: {dict(zip(*np.unique(y_train_sm, return_counts=True)))}")

# Phase 4: Build Neural Network (MLP)

Architecture:
- Input layer
- 3 hidden layers (ReLU activation)
- Dropout layers to prevent overfitting
- Output layer with Softmax (3 classes)

Loss: Sparse Categorical Crossentropy  
Optimizer: Adam

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train_sm.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax')  # 3 classes: fatal/minor/serious
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## Model Training

- Trains the model for 30 epochs
- Uses 10% validation split
- Tracks accuracy and loss

Expected:
Accuracy improves steadily during training

In [ ]:
history = model.fit(
    X_train_sm, y_train_sm,
    validation_split=0.1,
    epochs=30,
    batch_size=64,
    verbose=1
)
print("Training complete!")

## Training Performance Visualization

Plots:
- Accuracy vs Epochs
- Loss vs Epochs

Used to check:
- Learning trend
- Overfitting/underfitting

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

ax1.plot(history.history['accuracy'], label='Train', color='#3498db')
ax1.plot(history.history['val_accuracy'], label='Validation', color='#e74c3c')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['loss'], label='Train', color='#3498db')
ax2.plot(history.history['val_loss'], label='Validation', color='#e74c3c')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Phase 5: Model Evaluation

Evaluates model performance using:
- Classification Report (Precision, Recall, F1-score)
- Confusion Matrix

Helps understand prediction quality for each class

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print("Classification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Fatal', 'Minor Injury', 'Serious Injury']))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm,
       display_labels=['Fatal', 'Minor', 'Serious'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix — RTA Severity Predictor')
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Phase 6: SHAP Explainability

Uses SHAP to interpret model predictions.

Steps:
- Select background dataset
- Initialize KernelExplainer
- Compute SHAP values for test samples

In [ ]:
feature_names = list(X.columns)

# Use 150 background samples — fast enough for this small dataset
background = X_train_sm[np.random.choice(
    X_train_sm.shape[0], 150, replace=False)]

explainer = shap.KernelExplainer(model.predict, background)

# Explain 80 test samples
X_test_sample = X_test[:80]
shap_values = explainer.shap_values(X_test_sample)

print("SHAP values computed!")

## SHAP Feature Importance (Bar Plot)

Shows overall importance of features influencing predictions.

Focus: Minor Injury class (most frequent)

In [ ]:
shap.summary_plot(
    shap_values[1],
    X_test_sample,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)
plt.title('Top Features Influencing Minor Injury Prediction')
plt.savefig('outputs/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## SHAP Beeswarm Plot

Displays:
- Feature impact distribution
- Direction of influence (positive/negative)

Red → increases prediction  
Blue → decreases prediction

In [ ]:
shap.summary_plot(
    shap_values[1],
    X_test_sample,
    feature_names=feature_names,
    show=False
)
plt.savefig('outputs/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## SHAP Waterfall Plot (Single Prediction)

Explains one prediction step-by-step:
- Base value → final prediction
- Shows contribution of each feature

In [ ]:
explanation = shap.Explanation(
    values       = shap_values[1][0],
    base_values  = explainer.expected_value[1],
    data         = X_test_sample[0],
    feature_names= feature_names
)
shap.plots.waterfall(explanation, show=False)
plt.savefig('outputs/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()